# Statistical Evaluation of Uniformity Features for Readability

This notebook demonstrates the Uniformity Principle hypothesis for readability assessment. The experiment evaluates whether adding variance/uniformity measures of linguistic properties improves readability prediction beyond traditional average-based features.

## What This Notebook Does

Using a dataset of sentences with readability scores, we:
1. Compute linguistic features (word length, syllables, frequency) and their uniformity (coefficient of variation)
2. Run statistical tests to evaluate if uniformity features improve prediction:
   - Paired bootstrap MSE test
   - Bootstrap confidence intervals for regression coefficients
   - Cross-validation comparison
   - Effect size analysis
   - Ablation study

## Dataset

The demo uses a curated subset of the WeeBIT dataset (~50 sentences) for fast execution.


In [ ]:
# Install dependencies - works on both Colab and local Jupyter
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install)
_pip('loguru')
_pip('pronouncing')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

print('Install complete!')


In [ ]:
# Imports - copied from original method.py
from loguru import logger
from pathlib import Path
import json
import sys
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import pronouncing
import re
import nltk
from collections import Counter
import gc
import matplotlib.pyplot as plt

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)

# Setup logger (simplified for notebook)
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

print('Imports complete!')


In [ ]:
# Data loading helper - GitHub URL with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-fea63a-the-uniformity-principle-how-within-sent/main/round-2/experiment-2/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL or local file."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub fetch failed: {e}")
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

print('load_data() function defined!')


In [ ]:
# Load the demo data
data = load_data()

# Extract sentences and scores
sentences, scores, sources = [], [], []
for dataset in data['datasets']:
    for ex in dataset['examples']:
        sentences.append(ex['input'])
        scores.append(float(ex['output']))
        sources.append(dataset['dataset'])

print(f"Loaded {len(sentences)} sentences from {len(data['datasets'])} datasets")
print(f"Dataset sources: {set(sources)}")
print(f"Score range: {min(scores):.2f} - {max(scores):.2f}")

# Convert to numpy arrays
y = np.array(scores)
sources = np.array(sources)


## Configuration

Set tunable parameters for the experiments. For the demo, we use MINIMAL values:
- `N_BOOTSTRAP`: Number of bootstrap samples (original: 10000, demo: 200)
- `N_SPLITS`: Number of CV folds (original: 5, demo: 3)

Increase these for more accurate results (at the cost of runtime).


In [ ]:
# Configuration - MINIMAL values for fast demo
N_BOOTSTRAP = 200      # Original: 10000 - reduce for demo speed
N_SPLITS = 3           # Original: 5 - CV folds

print(f"Config: N_BOOTSTRAP={N_BOOTSTRAP}, N_SPLITS={N_SPLITS}")


## Feature Computation

We compute two types of linguistic features:

1. **Average features** (traditional readability measures):
   - `avg_word_length`: Mean word length in characters
   - `avg_syllables`: Mean syllables per word
   - `avg_frequency`: Mean log word frequency
   - `sentence_length`: Number of words

2. **Uniformity features** (coefficient of variation):
   - `cv_word_length`: Std/Avg of word lengths (higher = more varied)
   - `cv_syllables`: Std/Avg of syllables (higher = more varied)
   - `cv_frequency`: Std/Avg of word frequencies (higher = more varied)


In [ ]:
# Feature computation functions (from original method.py)

def count_syllables(word):
    """Count syllables using CMUdict with heuristic fallback."""
    word = word.lower().strip()
    if not word:
        return 1
    
    # Try CMUdict first
    phones = pronouncing.phones_for_word(word)
    if phones:
        return len([p for p in phones[0].split() if any(c.isdigit() for c in p)])
    
    # Heuristic fallback: count vowel groups
    vowels = 'aeiouy'
    count = 0
    prev_was_vowel = False
    
    for i, char in enumerate(word):
        is_vowel = char in vowels
        if char == 'y' and i == len(word) - 1 and len(word) > 1:
            is_vowel = True
        if is_vowel and not prev_was_vowel:
            count += 1
        prev_was_vowel = is_vowel
    
    # Silent 'e' adjustment
    if word.endswith('e') and count > 1:
        count -= 1
    
    return max(1, count)


def get_word_frequency(word, freq_dict):
    """Get log-transformed word frequency."""
    return freq_dict.get(word.lower(), 0)


def build_frequency_dict():
    """Build frequency dictionary from NLTK Gutenberg corpus."""
    logger.info("Building word frequency dictionary from NLTK Gutenberg corpus")
    try:
        from nltk.corpus import gutenberg
        words = gutenberg.words()
        freq = Counter(w.lower() for w in words)
        total = sum(freq.values())
        freq_dict = {w: np.log1p(c) / total for w, c in freq.items()}
        logger.info(f"Built frequency dict with {len(freq_dict)} words")
        return freq_dict
    except Exception as e:
        logger.warning(f"Failed to build frequency dict: {e}")
        return {}


def compute_features(sentences, freq_dict):
    """Compute all features for a list of sentences."""
    logger.info(f"Computing features for {len(sentences)} sentences")
    
    features_list = []
    for i, sent in enumerate(sentences):
        if i % 10 == 0:
            logger.info(f"Processing sentence {i}/{len(sentences)}")
        
        # Tokenize words
        words = nltk.word_tokenize(sent)
        words = [w.lower() for w in words if w.isalpha()]
        
        if not words:
            features_list.append({
                'avg_word_length': 0,
                'avg_syllables': 0,
                'avg_frequency': 0,
                'cv_word_length': 0,
                'cv_syllables': 0,
                'cv_frequency': 0,
                'sentence_length': 0
            })
            continue
        
        # Compute word-level features
        word_lengths = [len(w) for w in words]
        syllables = [count_syllables(w) for w in words]
        frequencies = [get_word_frequency(w, freq_dict) for w in words]
        
        # Average features
        avg_word_length = np.mean(word_lengths)
        avg_syllables = np.mean(syllables)
        avg_frequency = np.mean(frequencies) if frequencies else 0
        
        # Uniformity features (coefficient of variation)
        cv_word_length = np.std(word_lengths) / (avg_word_length + 1e-10)
        cv_syllables = np.std(syllables) / (avg_syllables + 1e-10)
        cv_frequency = np.std(frequencies) / (avg_frequency + 1e-10) if avg_frequency > 0 else 0
        
        # Sentence length
        sentence_length = len(words)
        
        features_list.append({
            'avg_word_length': avg_word_length,
            'avg_syllables': avg_syllables,
            'avg_frequency': avg_frequency,
            'cv_word_length': cv_word_length,
            'cv_syllables': cv_syllables,
            'cv_frequency': cv_frequency,
            'sentence_length': sentence_length
        })
    
    return pd.DataFrame(features_list)

print('Feature computation functions defined!')


In [ ]:
# Build frequency dictionary and compute features
freq_dict = build_frequency_dict()
X = compute_features(sentences, freq_dict)

print(f"Computed features shape: {X.shape}")
print(f"Feature columns: {list(X.columns)}")
print()
print("First few rows:")
print(X.head())


## Experiment 1: Paired Bootstrap MSE Test

This test evaluates whether adding uniformity features reduces mean squared error (MSE) in readability prediction.

We compare:
- **Model A**: Average features only (baseline)
- **Model B**: Average + Uniformity features (combined)

Using bootstrap sampling to compute the distribution of MSE differences.


In [ ]:
# Experiment 1: Paired Bootstrap MSE Test

def paired_bootstrap_mse_test(X, y, n_bootstrap=200):
    """Paired bootstrap test for MSE reduction with uniformity features."""
    logger.info(f"Running paired bootstrap MSE test with {n_bootstrap} samples")
    
    np.random.seed(42)
    n = len(y)
    
    avg_feats = ['avg_word_length', 'avg_syllables', 'avg_frequency', 'sentence_length']
    unif_feats = ['cv_word_length', 'cv_syllables', 'cv_frequency']
    combined = avg_feats + unif_feats
    
    mse_diffs = []
    
    for b in range(n_bootstrap):
        if b % 50 == 0:
            logger.info(f"Bootstrap sample {b}/{n_bootstrap}")
        
        idx = np.random.choice(n, n, replace=True)
        oob = np.setdiff1d(np.arange(n), idx)
        if len(oob) < 2:
            continue
        
        # Average features only model
        sa = StandardScaler().fit(X.loc[idx, avg_feats])
        X_train_A = sa.transform(X.loc[idx, avg_feats])
        X_test_A = sa.transform(X.loc[oob, avg_feats])
        mA = Ridge(1.0, random_state=42).fit(X_train_A, y[idx])
        mse_A = mean_squared_error(y[oob], mA.predict(X_test_A))
        
        # Combined model
        sb = StandardScaler().fit(X.loc[idx, combined])
        X_train_B = sb.transform(X.loc[idx, combined])
        X_test_B = sb.transform(X.loc[oob, combined])
        mB = Ridge(1.0, random_state=42).fit(X_train_B, y[idx])
        mse_B = mean_squared_error(y[oob], mB.predict(X_test_B))
        
        mse_diffs.append(mse_A - mse_B)
    
    mse_diffs = np.array(mse_diffs)
    baseline_mse = np.mean((y - np.mean(y))**2)
    
    return {
        'p_value_one_sided': float(np.mean(mse_diffs <= 0)),
        'mse_reduction_mean': float(np.mean(mse_diffs)),
        'mse_reduction_pct': float((np.mean(mse_diffs) / baseline_mse) * 100) if baseline_mse > 0 else 0,
        'n_bootstrap': len(mse_diffs),
        'mse_diffs': mse_diffs
    }


# Run Experiment 1
print("="*60)
print("Experiment 1: Paired Bootstrap MSE Test")
print("="*60)

result_exp1 = paired_bootstrap_mse_test(X, y, n_bootstrap=N_BOOTSTRAP)

print(f"\nResults:")
print(f"  MSE reduction (mean): {result_exp1['mse_reduction_mean']:.6f}")
print(f"  MSE reduction (%):     {result_exp1['mse_reduction_pct']:.2f}%")
print(f"  P-value (one-sided):   {result_exp1['p_value_one_sided']:.4f}")
print(f"  Valid bootstrap samples: {result_exp1['n_bootstrap']}")


## Experiment 2: Bootstrap Coefficient Confidence Intervals

We compute bootstrap 95% confidence intervals for Ridge regression coefficients to identify which features are significant predictors of readability.


In [ ]:
# Experiment 2: Bootstrap Coefficient CI

def bootstrap_coef_ci(X, y, n_bootstrap=200):
    """Bootstrap 95% confidence intervals for Ridge regression coefficients."""
    logger.info(f"Computing bootstrap coefficient CI with {n_bootstrap} samples")
    
    np.random.seed(42)
    n, p = len(y), X.shape[1]
    coefs = np.zeros((n_bootstrap, p))
    
    for b in range(n_bootstrap):
        if b % 50 == 0:
            logger.info(f"Bootstrap sample {b}/{n_bootstrap}")
        
        idx = np.random.choice(n, n, replace=True)
        scaler = StandardScaler()
        Xs = scaler.fit_transform(X.iloc[idx])
        model = Ridge(1.0, random_state=42).fit(Xs, y[idx])
        coefs[b] = model.coef_
    
    results = []
    for i, f in enumerate(X.columns):
        c = coefs[:, i]
        ci_low = float(np.percentile(c, 2.5))
        ci_high = float(np.percentile(c, 97.5))
        mean_coef = float(np.mean(c))
        
        results.append({
            'feature': f,
            'mean_coef': mean_coef,
            'ci_95_lower': ci_low,
            'ci_95_upper': ci_high,
            'significant': (ci_low > 0) if mean_coef > 0 else (ci_high < 0)
        })
    
    return results


# Run Experiment 2
print("="*60)
print("Experiment 2: Bootstrap Coefficient CI")
print("="*60)

combined_feats = ['avg_word_length', 'avg_syllables', 'avg_frequency', 'sentence_length',
                 'cv_word_length', 'cv_syllables', 'cv_frequency']
result_exp2 = bootstrap_coef_ci(X[combined_feats], y, n_bootstrap=N_BOOTSTRAP)

print(f"\nRidge Regression Coefficients (95% CI):")
print(f"{'Feature':<20} {'Coef':>10} {'95% CI':>25} {'Significant':>12}")
print("-"*70)
for r in result_exp2:
    ci_str = f"[{r['ci_95_lower']:.4f}, {r['ci_95_upper']:.4f}]"
    sig_str = "YES" if r['significant'] else "no"
    print(f"{r['feature']:<20} {r['mean_coef']:>10.4f} {ci_str:>25} {sig_str:>12}")


## Experiment 3: Cross-Validation Comparison

We use proper K-fold cross-validation to compare the predictive performance of:
- **Average features only** (baseline)
- **Combined features** (average + uniformity)

The metric is R2 (coefficient of determination).


In [ ]:
# Experiment 3: Cross-Validation Comparison

def cv_evaluate(X, y, n_splits=3):
    """K-fold cross-validation with proper train/test separation."""
    n_samples = len(X)
    actual_splits = min(n_splits, n_samples - 1)
    if actual_splits < 2:
        from sklearn.model_selection import train_test_split
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
        scaler = StandardScaler().fit(X_train)
        model = Ridge(1.0, random_state=42).fit(scaler.transform(X_train), y_train)
        y_pred = model.predict(scaler.transform(X_test))
        r2 = r2_score(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        return {'test_r2_mean': float(r2), 'test_mse_mean': float(mse)}
    
    kf = KFold(actual_splits, shuffle=True, random_state=42)
    r2_folds, mse_folds = [], []
    
    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
        scaler = StandardScaler().fit(X.iloc[train_idx])
        model = Ridge(1.0, random_state=42).fit(scaler.transform(X.iloc[train_idx]), y[train_idx])
        y_pred = model.predict(scaler.transform(X.iloc[test_idx]))
        
        r2_folds.append(r2_score(y[test_idx], y_pred))
        mse_folds.append(mean_squared_error(y[test_idx], y_pred))
    
    return {
        'test_r2_mean': float(np.mean(r2_folds)),
        'test_r2_sd': float(np.std(r2_folds)),
        'test_mse_mean': float(np.mean(mse_folds)),
        'test_mse_sd': float(np.std(mse_folds))
    }


# Run Experiment 3
print("="*60)
print("Experiment 3: Cross-Validation Comparison")
print("="*60)

avg_feats = ['avg_word_length', 'avg_syllables', 'avg_frequency', 'sentence_length']
combined_feats = avg_feats + ['cv_word_length', 'cv_syllables', 'cv_frequency']

result_cv_avg = cv_evaluate(X[avg_feats], y, n_splits=N_SPLITS)
result_cv_combined = cv_evaluate(X[combined_feats], y, n_splits=N_SPLITS)

print(f"\nResults:")
print(f"{'Model':<25} {'R2 (mean)':>12} {'R2 (SD)':>12} {'MSE (mean)':>12}")
print("-"*65)
print(f"{'Average features only':<25} {result_cv_avg['test_r2_mean']:>12.4f} {result_cv_avg.get('test_r2_sd', 0):>12.4f} {result_cv_avg['test_mse_mean']:>12.6f}")
print(f"{'Combined (avg+uniform)':<25} {result_cv_combined['test_r2_mean']:>12.4f} {result_cv_combined.get('test_r2_sd', 0):>12.4f} {result_cv_combined['test_mse_mean']:>12.6f}")
print(f"\nR2 improvement: {result_cv_combined['test_r2_mean'] - result_cv_avg['test_r2_mean']:.4f}")


## Experiment 4: Effect Size Analysis

We compute the effect size (Cohen's d) for the improvement from adding uniformity features. This tells us the practical significance of the result.


In [ ]:
# Experiment 4: Effect Size Analysis

def effect_size_analysis(X_avg, X_combined, y, n_bootstrap=200):
    """Compute effect size analysis."""
    logger.info("Computing effect size analysis")
    
    # Compute point estimates using CV
    cv_avg = cv_evaluate(X_avg, y)
    cv_combined = cv_evaluate(X_combined, y)
    
    r2_avg = cv_avg['test_r2_mean']
    r2_combined = cv_combined['test_r2_mean']
    r2_diff = r2_combined - r2_avg
    
    # Convert R2 to correlation
    r_avg = np.sqrt(max(0, r2_avg))
    r_combined_corr = np.sqrt(max(0, r2_combined))
    
    # Cohen's d approximation
    if 0 < r_combined_corr < 1:
        cohens_d = 2 * r_combined_corr / np.sqrt(1 - r_combined_corr**2)
    else:
        cohens_d = 0
    
    # Interpretation
    if abs(cohens_d) < 0.2:
        interpretation = "negligible"
    elif abs(cohens_d) < 0.5:
        interpretation = "small"
    elif abs(cohens_d) < 0.8:
        interpretation = "medium"
    else:
        interpretation = "large"
    
    return {
        'r2_avg': float(r2_avg),
        'r2_combined': float(r2_combined),
        'r2_difference': float(r2_diff),
        'correlation_combined': float(r_combined_corr),
        'cohens_d': float(cohens_d),
        'effect_interpretation': interpretation
    }


# Run Experiment 4
print("="*60)
print("Experiment 4: Effect Size Analysis")
print("="*60)

result_exp4 = effect_size_analysis(X[avg_feats], X[combined_feats], y, n_bootstrap=N_BOOTSTRAP)

print(f"\nResults:")
print(f"  R2 (average only):      {result_exp4['r2_avg']:.4f}")
print(f"  R2 (combined):          {result_exp4['r2_combined']:.4f}")
print(f"  R2 difference:          {result_exp4['r2_difference']:.4f}")
print(f"  Correlation (combined):  {result_exp4['correlation_combined']:.4f}")
print(f"  Cohen's d:              {result_exp4['cohens_d']:.4f}")
print(f"  Effect interpretation:   {result_exp4['effect_interpretation']}")


## Experiment 5: Ablation Study

We perform an ablation study to understand the contribution of each uniformity feature:
- **Add-one-in**: Start with average features, add one uniformity feature at a time
- **Remove-one-out**: Start with all features, remove one uniformity feature at a time


In [ ]:
# Experiment 5: Ablation Study

def ablation_study(X, y):
    """Add-one-in and remove-one-out uniformity feature analysis."""
    logger.info("Running ablation study")
    
    avg_feats = ['avg_word_length', 'avg_syllables', 'avg_frequency', 'sentence_length']
    unif_feats = ['cv_word_length', 'cv_syllables', 'cv_frequency']
    
    results = []
    
    # Baseline: average features only
    baseline_r2 = cv_evaluate(X[avg_feats], y)['test_r2_mean']
    results.append({
        'condition': 'baseline_avg_only',
        'features': avg_feats.copy(),
        'test_r2': baseline_r2
    })
    
    # Add-one-in: average + one uniformity feature at a time
    for uf in unif_feats:
        feats = avg_feats + [uf]
        r2 = cv_evaluate(X[feats], y)['test_r2_mean']
        results.append({
            'condition': f'add_{uf}',
            'features': feats.copy(),
            'test_r2': r2,
            'r2_improvement': r2 - baseline_r2
        })
    
    # Combined model
    combined_feats = avg_feats + unif_feats
    combined_r2 = cv_evaluate(X[combined_feats], y)['test_r2_mean']
    results.append({
        'condition': 'combined_all',
        'features': combined_feats.copy(),
        'test_r2': combined_r2,
        'r2_improvement': combined_r2 - baseline_r2
    })
    
    return results


# Run Experiment 5
print("="*60)
print("Experiment 5: Ablation Study")
print("="*60)

result_exp5 = ablation_study(X, y)

print(f"\nResults:")
print(f"{'Condition':<30} {'R2':>10} {'Improvement':>15}")
print("-"*60)
for r in result_exp5:
    imp_str = f"{r.get('r2_improvement', 0):+.4f}" if 'r2_improvement' in r else "-"
    print(f"{r['condition']:<30} {r['test_r2']:>10.4f} {imp_str:>15}")


## Visualization of Results

Let's visualize the key findings:
1. Coefficient magnitudes and significance
2. R2 comparison between models
3. Ablation study results


In [ ]:
# Visualization of key results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Coefficient magnitudes
coefs = [r['mean_coef'] for r in result_exp2]
features = [r['feature'] for r in result_exp2]
colors = ['red' if not r['significant'] else 'green' for r in result_exp2]

axes[0].barh(features, coefs, color=colors)
axes[0].axvline(x=0, color='black', linestyle='-', alpha=0.3)
axes[0].set_xlabel('Coefficient Value')
axes[0].set_title('Ridge Coefficients (green=significant)')

# Plot 2: R2 comparison
models = ['Average\nOnly', 'Combined\n(avg+unif)']
r2_values = [result_cv_avg['test_r2_mean'], result_cv_combined['test_r2_mean']]

axes[1].bar(models, r2_values, color=['blue', 'orange'])
axes[1].set_ylabel('R2')
axes[1].set_title('Cross-Validation R2 Comparison')
axes[1].set_ylim([0, max(r2_values) * 1.2])
for i, v in enumerate(r2_values):
    axes[1].text(i, v + 0.01, f'{v:.4f}', ha='center')

# Plot 3: Ablation results
conditions = [r['condition'].replace('baseline_', '').replace('add_', '+').replace('combined_', 'all\n') for r in result_exp5]
r2_ablation = [r['test_r2'] for r in result_exp5]

axes[2].bar(conditions, r2_ablation, color='purple')
axes[2].set_ylabel('R2')
axes[2].set_title('Ablation Study Results')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Print summary table
print("\n" + "="*70)
print("SUMMARY OF RESULTS")
print("="*70)
print(f"\n1. Bootstrap MSE Test:")
print(f"   MSE Reduction: {result_exp1['mse_reduction_pct']:.2f}% (p={result_exp1['p_value_one_sided']:.4f})")
print(f"\n2. Significant Features (95% CI excludes 0):")
for r in result_exp2:
    if r['significant']:
        print(f"   - {r['feature']}")
print(f"\n3. Cross-Validation R2 Improvement: {result_cv_combined['test_r2_mean'] - result_cv_avg['test_r2_mean']:.4f}")
print(f"\n4. Effect Size: Cohen's d = {result_exp4['cohens_d']:.2f} ({result_exp4['effect_interpretation']} effect)")
